# Customer Churn — Hyperparameter Tuning

## Objective

Improve the baseline customer churn models by tuning their hyperparameters using cross-validation.

## Models

- Logistic Regression
- Random Forest

## Evaluation Strategy

- 5-fold Stratified Cross-Validation
- Primary tuning metric: ROC-AUC
- Test set remains completely untouched during tuning

## Feature Set

The reduced 20-feature dataset identified during feature investigation is used.

Geographic features removed:

- City
- Lat Long
- Zip Code
- Latitude
- Longitude

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,confusion_matrix)
import warnings
warnings.filterwarnings("ignore")

In [3]:
data = pd.read_csv("../data/raw/telco_customer_churn.csv")
print("Dataset shape:", data.shape)
data.head()

Dataset shape: (7043, 33)


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [4]:
target_column = "Churn Label"

X = data.drop(columns=[target_column])
y = data[target_column]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

X shape: (7043, 32)
y shape: (7043,)

Target distribution:
Churn Label
No     5174
Yes    1869
Name: count, dtype: int64


In [5]:
selected_features = ["Tenure Months","Monthly Charges","Total Charges","CLTV","Gender","Senior Citizen","Partner","Dependents","Phone Service","Multiple Lines","Internet Service","Online Security","Online Backup","Device Protection","Tech Support","Streaming TV","Streaming Movies","Contract","Paperless Billing","Payment Method"]
X = X[selected_features]

print("Selected feature count:", len(selected_features))
print("X shape:", X.shape)

Selected feature count: 20
X shape: (7043, 20)


In [6]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numerical features: 3
['Tenure Months', 'Monthly Charges', 'CLTV']

Categorical features: 17
['Total Charges', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']


In [7]:
def create_preprocessor():
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ))
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features)
        ]
    )

    return preprocessor

In [8]:
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [9]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", create_preprocessor()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

In [10]:
logistic_param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__class_weight": [None, "balanced"]
}

In [11]:
logistic_grid = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_param_grid,
    scoring="roc_auc",
    cv=cv_strategy,
    n_jobs=-1,
    return_train_score=True
)

print("Starting Logistic Regression tuning...")

logistic_grid.fit(X, y)

print("Logistic Regression tuning completed.")

Starting Logistic Regression tuning...
Logistic Regression tuning completed.


In [12]:
print("Best Logistic Regression parameters:")
print(logistic_grid.best_params_)

print("\nBest Cross-Validation ROC-AUC:")
print(logistic_grid.best_score_)

Best Logistic Regression parameters:
{'model__C': 1, 'model__class_weight': None}

Best Cross-Validation ROC-AUC:
0.8566497494710715


In [13]:
logistic_results = pd.DataFrame(logistic_grid.cv_results_)

logistic_results = logistic_results[
    [
        "param_model__C",
        "param_model__class_weight",
        "mean_test_score",
        "std_test_score",
        "mean_train_score"
    ]
].sort_values(
    by="mean_test_score",
    ascending=False
)

logistic_results

,param_model__C,param_model__class_weight,mean_test_score,std_test_score,mean_train_score
4,1.00,NaN,0.856650,0.009438,0.937166
5,1.00,balanced,0.856501,0.009444,0.946039
2,0.10,NaN,0.856240,0.009604,0.869586
3,0.10,balanced,0.856227,0.009656,0.871458
1,0.01,balanced,0.854676,0.009430,0.858577
0,0.01,NaN,0.854177,0.009388,0.857728
6,10.00,NaN,0.851458,0.008482,0.998547
7,10.00,balanced,0.850961,0.008389,0.998583


In [17]:
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", create_preprocessor()),
        ("model", RandomForestClassifier(
            random_state=42,
            n_jobs=1
        ))
    ]
)

In [18]:
random_forest_param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

In [19]:
random_forest_grid = GridSearchCV(
    estimator=random_forest_pipeline,
    param_grid=random_forest_param_grid,
    scoring="roc_auc",
    cv=cv_strategy,
    n_jobs=-1,
    return_train_score=True
)

print("Starting Random Forest tuning...")

random_forest_grid.fit(X, y)

print("Random Forest tuning completed.")

Starting Random Forest tuning...


Random Forest tuning completed.


In [20]:
print("Best Random Forest parameters:")
print(random_forest_grid.best_params_)

print("\nBest Cross-Validation ROC-AUC:")
print(random_forest_grid.best_score_)

Best Random Forest parameters:
{'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 400}

Best Cross-Validation ROC-AUC:
0.8503840118837884


In [21]:
random_forest_results = pd.DataFrame(
    random_forest_grid.cv_results_
)

random_forest_results = random_forest_results[
    [
        "param_model__n_estimators",
        "param_model__max_depth",
        "param_model__min_samples_split",
        "param_model__min_samples_leaf",
        "mean_test_score",
        "std_test_score",
        "mean_train_score"
    ]
].sort_values(
    by="mean_test_score",
    ascending=False
)

random_forest_results.head(10)

,param_model__n_estimators,param_model__max_depth,param_model__min_samples_split,param_model__min_samples_leaf,mean_test_score,std_test_score,mean_train_score
3,400,None,5,1,0.850384,0.005465,0.999141
2,200,None,5,1,0.850040,0.005063,0.998990
19,400,20,5,1,0.847699,0.008337,0.921272
17,400,20,2,1,0.847293,0.008549,0.932943
18,200,20,5,1,0.847274,0.008185,0.920611
16,200,20,2,1,0.847261,0.008634,0.932787
1,400,None,2,1,0.845671,0.006937,1.000000
0,200,None,2,1,0.845520,0.007678,1.000000
9,400,10,2,1,0.837599,0.008647,0.862525
8,200,10,2,1,0.837461,0.008758,0.861932


In [22]:
tuning_summary = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Best CV ROC-AUC": [
        logistic_grid.best_score_,
        random_forest_grid.best_score_
    ]
})

tuning_summary.sort_values(
    by="Best CV ROC-AUC",
    ascending=False
)

,Model,Best CV ROC-AUC
0,Logistic Regression,0.856650
1,Random Forest,0.850384


In [23]:
tuned_models = {
    "Logistic Regression": logistic_grid.best_estimator_,
    "Random Forest": random_forest_grid.best_estimator_
}

print("Tuned models prepared:")
for name in tuned_models:
    print("-", name)

Tuned models prepared:
- Logistic Regression
- Random Forest


## Step 10 Conclusion

Hyperparameter tuning was performed for Logistic Regression and Random Forest using 5-fold Stratified Cross-Validation.

ROC-AUC was used as the primary tuning metric because the customer churn target is imbalanced and the model's probability ranking is important.

The test set was not used during hyperparameter tuning.

The tuned models will be evaluated on the untouched test set in the next stage before selecting and persisting the final customer churn model.